In [3]:
import time
import psutil

class Pair:
    def __init__(self, item=0, utility=0):
        self.item = item
        self.utility = utility

    def __str__(self):
        return f"[{self.item},{self.utility}]"


class Element:
    def __init__(self, tid, iutils, rutils):
        self.tid = tid
        self.iutils = iutils
        self.rutils = rutils


class UtilityList:
    def __init__(self, item):
        self.item = item
        self.sumIutils = 0
        self.sumRutils = 0
        self.elements = []

    def addElement(self, element):
        self.elements.append(element)
        self.sumIutils += element.iutils
        self.sumRutils += element.rutils


class AlgoHUI_LIST_INS:
    def __init__(self):
        self.maxMemory = 0
        self.startTimestamp = 0
        self.endTimestamp = 0
        self.huiCount = 0
        self.totalTimeForAllRuns = 0
        self.totalCandidateCountForAllRuns = 0
        self.candidateCount = 0

        self.mapItemToTWU = {}
        self.mapItemToRank = {}
        self.writer = None
        self.mapEUCS = {}
        self.debug = False
        self.mapItemToUtilityList = {}
        self.listOfUtilityLists = []
        self.totalDBUtility = 0
        self.minUtility = 0
        self.BUFFERS_SIZE = 200
        self.itemsetBuffer = [0] * self.BUFFERS_SIZE
        self.currentTidForUtilityList = 0

    def runAlgorithm(self, input_path, output_path, minUtility, firstLine, lastLine):
        self.candidateCount = 0
        self.huiCount = 0
        self.itemsetBuffer = [0] * self.BUFFERS_SIZE
        self.maxMemory = 0

        self.writer = open(output_path, 'w')
        if self.mapEUCS is None:
            self.mapEUCS = {}
            self.listOfUtilityLists = []
            self.mapItemToRank = {}
            self.mapItemToUtilityList = {}

        self.startTimestamp = time.time()

        newItemsUtilityLists = []

        if self.mapItemToTWU is None:
            self.mapItemToTWU = {}

        with open(input_path, 'r') as file:
            tid = 0
            for line in file:
                if tid >= lastLine:
                    break

                if tid >= firstLine:
                    if line.startswith(('#', '%', '@')) or line.strip() == '':
                        continue

                    split = line.strip().split(":")
                    items = list(map(int, split[0].split(" ")))
                    transactionUtility = int(split[1])

                    for item in items:
                        twu = self.mapItemToTWU.get(item, 0) + transactionUtility
                        self.mapItemToTWU[item] = twu
                        if item not in self.mapItemToUtilityList:
                            uList = UtilityList(item)
                            self.mapItemToUtilityList[item] = uList
                            newItemsUtilityLists.append(uList)

                    self.totalDBUtility += transactionUtility

                tid += 1

        self.minUtility = minUtility

        newItemsUtilityLists.sort(key=lambda x: self.mapItemToTWU[x.item])

        for idx, uList in enumerate(newItemsUtilityLists):
            self.mapItemToRank[uList.item] = idx + 1

        self.listOfUtilityLists.extend(newItemsUtilityLists)

        with open(input_path, 'r') as file:
            tid = 0
            for line in file:
                if tid >= lastLine:
                    break

                if line.startswith(('#', '%', '@')) or line.strip() == '':
                    continue

                if tid >= firstLine:
                    self.currentTidForUtilityList += 1

                    split = line.strip().split(":")
                    items = list(map(int, split[0].split(" ")))
                    utilityValues = list(map(int, split[2].split(" ")))

                    remainingUtility = 0
                    newTWU = 0
                    revisedTransaction = []

                    for i in range(len(items)):
                        pair = Pair(items[i], utilityValues[i])
                        revisedTransaction.append(pair)
                        remainingUtility += pair.utility
                        newTWU += pair.utility

                    revisedTransaction.sort(key=lambda x: self.mapItemToRank[x.item])

                    for i in range(len(revisedTransaction)):
                        pair = revisedTransaction[i]
                        remainingUtility -= pair.utility
                        utilityListOfItem = self.mapItemToUtilityList[pair.item]
                        element = Element(self.currentTidForUtilityList, pair.utility, remainingUtility)
                        utilityListOfItem.addElement(element)

                        mapFMAPItem = self.mapEUCS.get(pair.item)
                        if mapFMAPItem is None:
                            mapFMAPItem = {}
                            self.mapEUCS[pair.item] = mapFMAPItem

                        for j in range(i + 1, len(revisedTransaction)):
                            pairAfter = revisedTransaction[j]
                            twuSum = mapFMAPItem.get(pairAfter.item, 0) + newTWU
                            mapFMAPItem[pairAfter.item] = twuSum

                tid += 1

        self.checkMemory()
        self.huiListIns(self.itemsetBuffer, 0, None, self.listOfUtilityLists, self.minUtility)
        self.checkMemory()

        self.writer.close()

        self.endTimestamp = time.time()
        self.totalTimeForAllRuns += (self.endTimestamp - self.startTimestamp)
        self.totalCandidateCountForAllRuns += self.candidateCount

    def checkMemory(self):
        currentMemory = psutil.Process().memory_info().rss / (1024 * 1024)
        if currentMemory > self.maxMemory:
            self.maxMemory = currentMemory

    def huiListIns(self, prefix, prefixLength, pUL, ULs, minUtility):
        for i in range(len(ULs)):
            X = ULs[i]

            if X.sumIutils >= minUtility:
                self.writeOut(prefix, prefixLength, X.item, X.sumIutils, len(X.elements))

            if X.sumIutils + X.sumRutils >= minUtility:
                exULs = []
                for j in range(i + 1, len(ULs)):
                    Y = ULs[j]

                    mapTWUF = self.mapEUCS.get(X.item)
                    if mapTWUF:
                        twuF = mapTWUF.get(Y.item)
                        if twuF is None or twuF < minUtility:
                            continue

                    self.candidateCount += 1
                    temp = self.construct(pUL, X, Y)
                    exULs.append(temp)

                self.itemsetBuffer[prefixLength] = X.item
                self.huiListIns(self.itemsetBuffer, prefixLength + 1, X, exULs, minUtility)

    def construct(self, P, px, py):
        pxyUL = UtilityList(py.item)
        for ex in px.elements:
            ey = self.findElementWithTID(py, ex.tid)
            if ey is None:
                continue

            if P is None:
                eXY = Element(ex.tid, ex.iutils + ey.iutils, ey.rutils)
                pxyUL.addElement(eXY)
            else:
                e = self.findElementWithTID(P, ex.tid)
                if e:
                    eXY = Element(ex.tid, ex.iutils + ey.iutils - e.iutils, ey.rutils)
                    pxyUL.addElement(eXY)

        return pxyUL

    def findElementWithTID(self, ulist, tid):
        for element in ulist.elements:
            if element.tid == tid:
                return element
        return None

    def writeOut(self, prefix, prefixLength, item, sumIutils, support):
        self.huiCount += 1
        buffer = ' '.join(map(str, prefix[:prefixLength])) + f' {item} #UTIL: {sumIutils}'
        self.writer.write(buffer + '\n')

    def printStats(self):
        print("=============  HUI-LIST_INS ALGORITHM - STATS =============")
        print(f" Transaction processed count : {self.currentTidForUtilityList}")
        print(f" Execution time ~ {self.endTimestamp - self.startTimestamp} ms")
        print(f" Memory ~ {self.maxMemory} MB")
        print(f" High-utility itemsets count : {self.huiCount}")
        print(f" Candidate count : {self.candidateCount}")
        print(f" minutil : {self.minUtility}")
        print("===================================================")
        print(f"TOTAL CANDIDATE COUNT FOR ALL RUNS: {self.totalCandidateCountForAllRuns} candidates")
        print(f"TOTAL TIME FOR ALL RUNS: {self.totalTimeForAllRuns} ms")
        print("===================================================")


In [4]:
import os

class MainTestHUI_LIST_INS:
    @staticmethod
    def main():
        # Set the output file path
        output = "./outputHUI-LIST-INSt.xt"

        # Initialize the algorithm
        algo = AlgoHUI_LIST_INS()

        # Set the minimum utility threshold
        min_utility = 30

        # 1) Apply the algorithm on a first file containing transactions
        print("1) Run the algorithm on the first file")

        input1 = MainTestHUI_LIST_INS.fileToPath("DB_UtilityIncremental1.txt")
        algo.runAlgorithm(input1, output, min_utility, 0, float('inf'))
        algo.printStats()

        # The result has been saved to the file output.txt

        # Print the number of HUIs found until now to the console
        print(f"NUMBER OF HUI FOUND: {algo.huiCount}")

        # 2) Apply the algorithm on a second file containing transactions
        print("\n2) Run the algorithm on the second file")

        # Applying the algorithm
        input2 = MainTestHUI_LIST_INS.fileToPath("DB_UtilityIncremental2.txt")
        # Uncomment the next line to run the algorithm on the second file
        # algo.runAlgorithm(input2, output, min_utility, 0, float('inf'))
        algo.printStats()

        # The result has been saved to the file output.txt, and has overwritten the previous result.

        # Print the number of HUIs found until now to the console
        print(f"NUMBER OF HUI FOUND: {algo.huiCount}")

    @staticmethod
    def fileToPath(filename):
        # Directly return the relative path
        return os.path.join(os.getcwd(), filename)


if __name__ == "__main__":
    MainTestHUI_LIST_INS.main()


1) Run the algorithm on the first file
=============  HUI-LIST_INS ALGORITHM - STATS =============
 Transaction processed count : 4
 Execution time ~ 0.005315542221069336 ms
 Memory ~ 69.83984375 MB
 High-utility itemsets count : 6
 Candidate count : 29
 minutil : 30
TOTAL CANDIDATE COUNT FOR ALL RUNS: 29 candidates
TOTAL TIME FOR ALL RUNS: 0.005315542221069336 ms
NUMBER OF HUI FOUND: 6

2) Run the algorithm on the second file
=============  HUI-LIST_INS ALGORITHM - STATS =============
 Transaction processed count : 4
 Execution time ~ 0.005315542221069336 ms
 Memory ~ 69.83984375 MB
 High-utility itemsets count : 6
 Candidate count : 29
 minutil : 30
TOTAL CANDIDATE COUNT FOR ALL RUNS: 29 candidates
TOTAL TIME FOR ALL RUNS: 0.005315542221069336 ms
NUMBER OF HUI FOUND: 6
